In [1]:
import os
import json
from dotenv import load_dotenv
from rich.console import Console

from openai import OpenAI
from agents import Agent, Runner, trace, function_tool

In [2]:
# List of TODO and COMPLETED #
todos = []
completed = []

load_dotenv(override=True)

def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)


def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

@function_tool
def create_todos(descriptions: list[str]) -> str:
    """Add new todos from a list of descriptions and return the full list"""
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

@function_tool
def mark_complete(index: int, completion_notes: str) -> str:
    """Mark complete the todo at the given position (starting from 1) and return the full list"""
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

tools = [create_todos, mark_complete]

openai = OpenAI()

question_generator_prompt = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question_generator_prompt}]
questonGeneratorResponse = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
    )

question = question = questonGeneratorResponse.choices[0].message.content

print(f"Generated Question: {question} \n\n")

print(tools)

print("\n\n")

todo_planner_executor_system_prompt = """
You are  a todo list planner and executor. For a given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
todoPlannerAndExecutorAgent = Agent(
    name="Todo List Generator and Executor",
    instructions= todo_planner_executor_system_prompt,
    tools= tools
)

print("Trace Available at: https://platform.openai.com/logs?api=traces")

result = ""

with trace("TO List Planner and Executor Trace"):
    result = await Runner.run(todoPlannerAndExecutorAgent, question)

print(f"\n\n {result}")

Generated Question: If two trains start from different stations 300 miles apart and travel toward each other at speeds of 40 mph and 60 mph respectively, but the faster train reduces its speed by 10 mph every 30 minutes while the slower train increases its speed by 5 mph every 20 minutes, how long will it take for the trains to meet? 


[FunctionTool(name='create_todos', description='Add new todos from a list of descriptions and return the full list', params_json_schema={'properties': {'descriptions': {'items': {'type': 'string'}, 'title': 'Descriptions', 'type': 'array'}}, 'required': ['descriptions'], 'title': 'create_todos_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001494BD39F80>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='mark_complete', description='Mark complete the todo at the given positi

Todo #1: Estimate how the speeds of the trains change over time.
Todo #2: Calculate the distance each train covers each interval before their speeds change again.
Todo #3: Add up the distances covered each interval until their combined distance equals or exceeds 300 miles.
Todo #4: Sum up the time intervals taken to cover the total distance.

The faster train starts at 60 mph and reduces by 10 mph every 30 minutes (0.5 hr). The slower train starts at 40 
mph and increases by 5 mph every 20 minutes (1/3 hr). Track their speeds at each interval to use in distance 
calculations.

Todo #1: Estimate how the speeds of the trains change over time.
Todo #2: Calculate the distance each train covers each interval before their speeds change again.
Todo #3: Add up the distances covered each interval until their combined distance equals or exceeds 300 miles.
Todo #4: Sum up the time intervals taken to cover the total distance.

Break the timeline into intervals based on the shortest change event: 20 minutes (1/3 hr). For each of these: - 
Train A (40 mph initial) increases by 5 mph every interval. - Train B (60 mph initial) decreases by 10 mph every 30
minutes (so every 2 intervals). For each 20-min interval, calculate Train A and Train B's speeds over that segment 
and the total distance they cover together.

...

Sum up the distances over 20-minute intervals using the changing speeds: 
1st interval (0-20 min): Train A: 40 mph, Train B: 60 mph = 100 mph combined x 1/3 hr = 33.33 mi
2nd (20-40 min): Train A: 45 mph, Train B: 60 mph = 105 mph x 1/3 = 35 mi (Total: 68.33 mi)
3rd (40-60 min): Train A: 50 mph, Train B: 50 mph (since Train B now decreases speed) = 100 mph x 1/3 = 33.33 mi 
(Total: 101.66 mi)
4th (60-80 min): Train A: 55 mph, Train B: 50 mph = 105 mph x 1/3 = 35 mi (Total: 136.66 mi)
5th (80-100 min): Train A: 60 mph, Train B: 40 mph = 100 mph x 1/3 = 33.33 mi (Total: 169.99 mi)
6th (100-120 min): Train A: 65 mph, Train B: 40 mph = 105 mph x 1/3 = 35 mi (Total: 204.99 mi)
7th (120-140 min): Train A: 70 mph, Train B: 30 mph = 100 mph x 1/3 = 33.33 mi (Total: 238.32 mi)
8th (140-160 min): Train A: 75 mph, Train B: 30 mph = 105 mph x 1/3 = 35 mi (Total: 273.32 mi)
9th (160-180 min): Train A: 80 mph, Train B: 20 mph = 100 mph x 1/3 = 33.33 mi (Total: 306.65 mi)
They meet during the 9th interval. Previous total was 273.32 mi, so in 300 - 273.32 = 26.68 miles needed in 9th 
interval, and combined speed is 100 mph.

Todo #1: Estimate how the speeds of the trains change over time.
Todo #2: Calculate the distance each train covers each interval before their speeds change again.
Todo #3: Add up the distances covered each interval until their combined distance equals or exceeds 300 miles.
Todo #4: Sum up the time intervals taken to cover the total distance.

From previous step, after 8 intervals (160 min) they've covered 273.32 miles. In the 9th interval, they need to 
cover 26.68 mi at a combined 100 mph. 
Time required = 26.68 miles / 100 mph = 0.2668 hr ≈ 16 min. 
So, total time to meet = 160 + 16 = 176 minutes = 2 hr 56 min.

Todo #1: Estimate how the speeds of the trains change over time.
Todo #2: Calculate the distance each train covers each interval before their speeds change again.
Todo #3: Add up the distances covered each interval until their combined distance equals or exceeds 300 miles.
Todo #4: Sum up the time intervals taken to cover the total distance.



 RunResult:
- Last agent: Agent(name="Todo List Generator and Executor", ...)
- Final output (str):
    🟢 Solution: 
    
    - The two trains, starting 300 miles apart with their speeds changing at each interval, will meet after about **2 hours and 56 minutes**.
    
    **Breakdown of Approach:**
    - Analyzed speeds at each interval due to the different change rates.
    - Calculated the combined distance covered every 20 minutes.
    - Tracked the sum until just exceeding 300 miles.
    - Used the remaining distance in the last interval to find the precise meeting time.
    
    **Final answer:**  
    **The trains will meet after approximately 2 hours and 56 minutes.**
- 11 new item(s)
- 6 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)
